# TFT training on Colab — with HP search + Drive checkpointing

**Setup once:**
1. Mount Drive (Cell 2) and confirm the folder layout it creates.
2. Upload `dataset.parquet` and `dataset_meta.json` from your local `model/` directory to `/content/drive/MyDrive/dis_tft/data/` (drag-and-drop in the Files pane on the left, then move them into that folder).

**Resume after disconnect:** just re-run all cells. Optuna resumes from the SQLite db on Drive, finished checkpoints are skipped.

**Output:** `predictions_tft.parquet` lands in `/content/drive/MyDrive/dis_tft/artifacts/`. Download that to your local `model/artifacts_tft/` folder, then run `python3 model/ensemble.py && python3 model/evaluate.py` locally.

In [ ]:
!pip install -q pytorch-forecasting==1.0.0 lightning==2.1.4 optuna pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/dis_tft')
DATA_DIR = DRIVE_ROOT / 'data'
ART_DIR  = DRIVE_ROOT / 'artifacts'
OPTUNA_DB = DRIVE_ROOT / 'optuna_tft.db'
for d in (DATA_DIR, ART_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Drive ready. Upload dataset.parquet + dataset_meta.json to:', DATA_DIR)

In [ ]:
import torch, json, numpy as np, pandas as pd
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
df = pd.read_parquet(DATA_DIR / 'dataset.parquet')
META = json.loads((DATA_DIR / 'dataset_meta.json').read_text())
df['hour_utc'] = pd.to_datetime(df['hour_utc'], utc=True)
df = df.sort_values('hour_utc').reset_index(drop=True)
df['time_idx'] = np.arange(len(df), dtype=np.int64)
df['group'] = 'usd_rub'
print('rows:', len(df), 'horizons:', META['horizons'])

In [ ]:
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

QUANTILES = [0.1, 0.5, 0.9]
MAX_ENC = 48
MAX_PRED = max(META['horizons'])
TRAIN_CUT = META['splits']['train_pool_end']

TIME_KNOWN = ['time_idx', 'hour_of_day', 'day_of_week', 'is_peak', 'is_weekend', 'ref_rate']
TIME_UNKNOWN = ['turnover_usd', 'bank_rate', 'spread', 'spread_pct']

def make_ds(target):
    return TimeSeriesDataSet(
        df[df.time_idx <= TRAIN_CUT],
        time_idx='time_idx', target=target, group_ids=['group'],
        max_encoder_length=MAX_ENC, max_prediction_length=MAX_PRED,
        static_categoricals=['group'],
        time_varying_known_reals=TIME_KNOWN,
        time_varying_unknown_reals=TIME_UNKNOWN,
        target_normalizer=GroupNormalizer(groups=['group'], transformation='softplus'),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
        allow_missing_timesteps=True,
    )

## HP search — uses Optuna with SQLite study on Drive (resumable)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
STORAGE = f'sqlite:///{OPTUNA_DB}'
N_TRIALS_TFT = 20  # per the proposal §4.4

def best_params_path(name):
    return ART_DIR / f'best_{name}.json'

def search_hp(target, name, n_trials=N_TRIALS_TFT, max_epochs=15):
    if best_params_path(name).exists():
        print(f'✓ best params for {name} already saved; skipping search')
        return json.loads(best_params_path(name).read_text())
    train_ds = make_ds(target)
    val_ds = TimeSeriesDataSet.from_dataset(train_ds, df, predict=False, stop_randomization=True)
    train_dl = train_ds.to_dataloader(train=True, batch_size=128, num_workers=2)
    val_dl = val_ds.to_dataloader(train=False, batch_size=256, num_workers=2)

    def objective(trial):
        params = {
            'hidden_size': trial.suggest_int('hidden_size', 16, 128, step=16),
            'attention_head_size': trial.suggest_int('attention_head_size', 1, 8),
            'dropout': trial.suggest_float('dropout', 0.05, 0.4),
            'hidden_continuous_size': trial.suggest_int('hidden_continuous_size', 8, 64, step=8),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3, log=True),
        }
        pl.seed_everything(7)
        model = TemporalFusionTransformer.from_dataset(
            train_ds, **params,
            loss=QuantileLoss(quantiles=QUANTILES), log_interval=0,
        )
        es = EarlyStopping(monitor='val_loss', patience=3, mode='min')
        trainer = pl.Trainer(max_epochs=max_epochs, accelerator='auto', gradient_clip_val=0.5,
                              callbacks=[es], enable_model_summary=False, enable_progress_bar=False, logger=False)
        trainer.fit(model, train_dl, val_dl)
        return float(trainer.callback_metrics.get('val_loss', float('inf')))

    study = optuna.create_study(
        study_name=f'tft_{name}', storage=STORAGE, load_if_exists=True,
        direction='minimize', sampler=optuna.samplers.TPESampler(seed=7),
    )
    completed = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    remaining = max(0, n_trials - completed)
    print(f'optuna {name!r}: {completed}/{n_trials} done, running {remaining} more')
    if remaining > 0:
        study.optimize(objective, n_trials=remaining,
                       callbacks=[lambda s, t: print(f'  trial {t.number+1} val={t.value:.4f}')])
    best = study.best_params
    best_params_path(name).write_text(json.dumps(best, indent=2))
    print(f'best {name}: {best}')
    return best

In [ ]:
def train_final(target, name, params, max_epochs=40):
    """Train final model with best HP, save checkpoint and predictions to Drive."""
    ckpt_path = ART_DIR / f'tft_{name}.ckpt'
    pred_path_partial = ART_DIR / f'preds_{name}.npy'
    decoder_path = ART_DIR / f'decoder_idx_{name}.npy'

    train_ds = make_ds(target)
    val_ds = TimeSeriesDataSet.from_dataset(train_ds, df, predict=False, stop_randomization=True)
    train_dl = train_ds.to_dataloader(train=True, batch_size=128, num_workers=2)
    val_dl = val_ds.to_dataloader(train=False, batch_size=256, num_workers=2)

    pl.seed_everything(7)
    model = TemporalFusionTransformer.from_dataset(
        train_ds, **params,
        loss=QuantileLoss(quantiles=QUANTILES), log_interval=0,
    )
    es = EarlyStopping(monitor='val_loss', patience=5, mode='min')
    ckpt = ModelCheckpoint(monitor='val_loss', mode='min', dirpath=str(ART_DIR), filename=f'tft_{name}')
    trainer = pl.Trainer(max_epochs=max_epochs, accelerator='auto', gradient_clip_val=0.5,
                          callbacks=[es, ckpt], enable_model_summary=False, logger=False)

    if ckpt_path.exists() and pred_path_partial.exists():
        print(f'✓ predictions for {name} already saved; reusing')
        return np.load(pred_path_partial), np.load(decoder_path)

    trainer.fit(model, train_dl, val_dl)

    # Predict on full series
    full_ds = TimeSeriesDataSet.from_dataset(train_ds, df, predict=True, stop_randomization=True)
    full_dl = full_ds.to_dataloader(train=False, batch_size=256, num_workers=2)
    preds = trainer.predict(model, full_dl, return_predictions=True)
    out = torch.cat(preds, dim=0).cpu().numpy()  # (n, pred_len, 3 quantiles)
    decoder_idx = full_ds.x_to_index(full_ds)['time_idx'].values
    np.save(pred_path_partial, out)
    np.save(decoder_path, decoder_idx)
    return out, decoder_idx

In [ ]:
# Direction A — turnover
best_A = search_hp('turnover_usd', 'A_turnover', n_trials=20, max_epochs=15)
out_A, decoder_idx_A = train_final('turnover_usd', 'A_turnover', best_A, max_epochs=40)
print('A shape:', out_A.shape)

In [ ]:
# Direction B — bank rate
best_B = search_hp('bank_rate', 'B_rate', n_trials=20, max_epochs=15)
out_B, decoder_idx_B = train_final('bank_rate', 'B_rate', best_B, max_epochs=40)
print('B shape:', out_B.shape)

In [ ]:
# Build predictions_tft.parquet on Drive
records = []
for h in META['horizons']:
    cum = out_A[:, :h, :].sum(axis=1)
    records.append(pd.DataFrame({
        'row_idx': decoder_idx_A,
        'p10': cum[:,0], 'p50': cum[:,1], 'p90': cum[:,2],
        'direction': 'A', 'horizon': h,
    }))
for h in META['horizons']:
    records.append(pd.DataFrame({
        'row_idx': decoder_idx_B,
        'p10': out_B[:,0,0], 'p50': out_B[:,0,1], 'p90': out_B[:,0,2],
        'direction': 'B', 'horizon': h,
    }))
final = pd.concat(records, ignore_index=True)
out_path = ART_DIR / 'predictions_tft.parquet'
final.to_parquet(out_path, index=False)
print('wrote', out_path, 'rows:', len(final))
print('Now download', out_path, 'to your local model/artifacts_tft/ folder.')